# Generate manuscript assets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yazanjer/An_Explainable_AI_Education/blob/main/notebooks/07_generate_manuscript_tables.ipynb)

**Answers:** Editor comment 10
**Estimated runtime:** 5 min · **Hardware:** CPU
**Quick mode:** set `QUICK_MODE = True` in the setup cell for a fast smoke test.

Emits every manuscript table as **both** `.tex` and `.csv`, then writes
`results/manifest.json` with a SHA256, git commit and config hash for every artefact.

No number in the manuscript is typed by hand.

---


In [ ]:
# --- Environment setup -------------------------------------------------
# Detects Colab, mounts Drive only when in Colab, installs pinned deps.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
QUICK_MODE = True   # set False for the full budget

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT = Path("/content/drive/MyDrive/An_Explainable_AI_Education")
    PROJECT.mkdir(parents=True, exist_ok=True)
    if not (PROJECT / "src").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/yazanjer/An_Explainable_AI_Education.git", str(PROJECT)],
                       check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT / "requirements.txt")], check=False)
else:
    PROJECT = Path(os.environ.get("VLPSO_PROJECT_ROOT", Path.cwd().parent))

os.environ["VLPSO_PROJECT_ROOT"] = str(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

from vlpso_xai.config import load_config, set_global_seeds, environment_report
cfg = load_config("quick" if QUICK_MODE else "default")
set_global_seeds(cfg.seed)
cfg.paths.mkdirs()
print("project root:", cfg.paths.root)
print("config:", cfg.config_path.name, "| hash:", cfg.hash()[:12])


In [ ]:
from vlpso_xai.reporting.tables import emit
from vlpso_xai.reporting.manifest import write_manifest
from vlpso_xai.config import environment_report

TAB = cfg.paths.tables
emit(flow, "table_sample_flow", TAB, caption="Sample flow, PISA 2018 Spain.")
emit(res.groupby("task")[["auc","w_auc","balanced_accuracy"]].mean().reset_index(),
     "table_nested_cv", TAB, caption="Nested cross-validation performance by task.")
emit(sel.groupby("method")[["auc","n_selected"]].mean().reset_index(),
     "table_selector_comparison", TAB, caption="Feature-selection method comparison.")
emit(imp.head(20), "table_global_shap", TAB, caption="Global SHAP importance with CIs.")
p = write_manifest(cfg.paths.results, config_hash=cfg.hash(),
                   generating_script="notebooks/07_generate_manuscript_tables.ipynb",
                   environment=environment_report())
print("manifest written:", p)